### 02 — Calidad y preparación del historial de running

#### Objetivo

Revisar el historial extraído de Garmin Connect y preparar una tabla
de carreras fiable para el análisis.

#### Punto de partida

Un archivo JSON original con 317 actividades obtenido mediante
el script de extracción.

#### Orden de trabajo

1. Cargar el archivo guardado, sin volver a consultar Garmin.
2. Revisar tamaño, campos, modalidades, duplicados y periodo temporal.
3. Comprobar la disponibilidad de los 54 campos seleccionados.
4. Revisar valores ausentes, unidades y posibles inconsistencias.
5. Aplicar y documentar las transformaciones necesarias.
6. Guardar los datos preparados sin modificar el archivo original.

Las decisiones tomadas con la muestra se comprobarán de nuevo
sobre el historial completo.

In [ ]:
# Permite leer el archivo JSON de Garmin.
import json

# Permite construir y comprobar rutas.
from pathlib import Path

# Permite trabajar con los datos como tablas.
import pandas as pd

# Mostramos desde qué carpeta se interpretan las rutas relativas.
print("Carpeta de trabajo:", Path.cwd())

In [ ]:
# Subimos a la raiz del proyecto y entramos a los datos
carpeta_raw = Path("../data/raw")

# Buscamos los archivos Json de running y ordenamos sus nombres.
archivos_json = sorted(carpeta_raw.glob("garmin_running_*.json"))

for archivo in archivos_json:
    print(archivo.name)

In [ ]:
# Construimos la ruta al archivo del historial completo.
ruta_historial = carpeta_raw / "garmin_running_20261031_200259.json"

# Abrimos el archivo en modo lectura y recuperamos el contenido.
with ruta_historial.open("r",encoding="utf-8") as archivo:
    actividades = json.load(archivo)

print("Actividades cargadas:",len(actividades))


In [ ]:
# Convertimos en un DataFrame. 
# Cada actividad una fila y cada campo una columna
df_original = pd.DataFrame(actividades)

# Consultamos tamaño
filas,columnas = df_original.shape

print("Actividades:", filas)
print("Columnas:", columnas)

In [ ]:
# Definimos cómo obtener la modalidad del diccionario de cada actividad.
def obtener_modalidad(tipo_actividad):
    return tipo_actividad["typeKey"]

# Aplicamos la funcion a cada valor de la columna activityType.
modalidades = df_original["activityType"].apply(obtener_modalidad)

# Contamos cuántas actividades hay de cada modalidad.
print(modalidades.value_counts(dropna=False))

#### Modalidades del historial

La extracción contiene 317 actividades:
- 282 de modalidad running.
- 35 de modalidad treadmill_running, correspondientes a carrera en cinta.

Conservaremos ambas modalidades y mantendremos su identificación.
Las métricas relacionadas con recorrido, desnivel o medición de distancia
se revisarán teniendo en cuenta la modalidad.


In [ ]:
# Contamos repeticiones de los ids.
ids_duplicados = df_original['activityId'].duplicated().sum()

# Contamos los ids ausentes.
ids_vacios = df_original["activityId"].isna().sum()

print("IDs duplicados:", ids_duplicados)
print("IDs vacíos:", ids_vacios)

#### Identificación y duplicados

No hay identificadores de actividad vacíos ni repetidos.
Las 317 actividades tienen un activityId distinto.

In [ ]:
# Seleccionamos las fechas locales y su referencia GMT.
campos_fecha = ["startTimeLocal","startTimeGMT"]

display(df_original[campos_fecha].head())

In [ ]:
# Convertimos el texto a fechas siguiendo el formato observado.
fechas_locales = pd.to_datetime(
    df_original["startTimeLocal"],
    format="%Y-%m-%d %H:%M:%S",
    errors="coerce"
)
# Contamos ausentes despues de la conversion
print("Fechas ausentes:",fechas_locales.isna().sum())

# Mostramos los extremos del historial
print("Fecha más antigua:",fechas_locales.min())
print("Fecha más reciente:",fechas_locales.max())

#### Periodo temporal

Las 317 fechas locales se han convertido correctamente, sin valores ausentes.

- Primera actividad: 2 de julio de 2023 a las 08:43:07.
- Última actividad: 31 de octubre de 2026 a las 09:29:33.

El periodo incluye carreras de exterior y de cinta.
Los extremos no implican que haya entrenamientos en todos los meses
ni que el registro sea continuo.

Conservamos las fechas originales sin modificar.

In [ ]:
# Guardamos los nombres de las columnas que queremos conservar.
columnas_conservar = [
    # Identificación, fechas, modalidad y récord personal.
    "activityId",
    "activityName",
    "startTimeLocal",
    "startTimeGMT",
    "activityType",
    "isPR",

    # Distancia, duración y velocidad.
    "distance",
    "duration",
    "elapsedDuration",
    "movingDuration",
    "averageSpeed",
    "maxSpeed",

    # Frecuencia cardiaca.
    "averageHR",
    "maxHR",

    # Desnivel y altitud.
    "elevationGain",
    "elevationLoss",
    "minElevation",
    "maxElevation",
    "avgElevation",

    # Cadencia y pasos.
    "averageRunningCadenceInStepsPerMinute",
    "maxRunningCadenceInStepsPerMinute",
    "steps",

    # Calorías y temperatura.
    "calories",
    "bmrCalories",
    "minTemperature",
    "maxTemperature",

    # Potencia.
    "avgPower",
    "maxPower",
    "normPower",

    # Efecto del entrenamiento.
    "aerobicTrainingEffect",
    "anaerobicTrainingEffect",
    "trainingEffectLabel",
    "aerobicTrainingEffectMessage",
    "anaerobicTrainingEffectMessage",

    # Dinámica de carrera.
    "avgStrideLength",
    "avgGroundContactTime",
    "avgVerticalOscillation",
    "avgVerticalRatio",

    # Tiempo en zonas de frecuencia cardiaca.
    "hrTimeInZone_1",
    "hrTimeInZone_2",
    "hrTimeInZone_3",
    "hrTimeInZone_4",
    "hrTimeInZone_5",

    # Tiempo en zonas de potencia.
    "powerTimeInZone_1",
    "powerTimeInZone_2",
    "powerTimeInZone_3",
    "powerTimeInZone_4",
    "powerTimeInZone_5",

    # Mejores tramos.
    "fastestSplit_1000",
    "fastestSplit_1609",
    "fastestSplit_5000",
    "fastestSplit_10000",
    "fastestSplit_21098",

    # VO₂ máximo: conservado, con interpretación pendiente.
    "vO2MaxValue"
]

# Contamos cuántos campos hemos seleccionado.
print("Campos seleccionados:", len(columnas_conservar))

# Buscamos cuales no existen en el historial completo.
columnas_faltantes = []

for columna in columnas_conservar:
    if columna not in df_original.columns:
        columnas_faltantes.append(columna)

print("Columnas faltantes:", columnas_faltantes)


In [ ]:
# Seleccionamos los 54 campos acordados y creamos una copia de trabajo.
df_running = df_original[columnas_conservar].copy()

# Resumimos los valores presentes y ausentes en cada campo.
resumen_disponibilidad = pd.DataFrame({
    "con_dato": df_running.notna().sum(),
    "vacios": df_running.isna().sum()
    })

# Calculamos que porcentaje de las actividades tiene el dato vacio.
resumen_disponibilidad["porcentaje_vacio"] = (
    resumen_disponibilidad["vacios"] / len(df_running) * 100
)

# Ordenamos de mayor a menor cantidad de vacíos.
resumen_disponibilidad = resumen_disponibilidad.sort_values(
    "vacios", ascending=False
)

# Mostramos los 54 campos y redondeamos los porcentajes para leerlos.
with pd.option_context("display.max_rows", None):
    display(resumen_disponibilidad.round(1))

In [ ]:
# Detectamos qué actividades no tienen dato de ascenso acumulado.
sin_desnivel = df_running["elevationGain"].isna()

# Cruzamos la modalidad con la presencia o ausencia de ese dato.
display(pd.crosstab(
    modalidades,
    sin_desnivel,
    rownames=["Modalidad"],
    colnames=["Ascenso ausente"]
))

In [ ]:
# Combinamos dos condiciones:
# modalidad running Y ausencia de ascenso.
condicion = (modalidades == "running") & sin_desnivel

# Mostramos los datos básicos de la actividad que cumple ambas.
display(df_running.loc[
    condicion,
    ["activityId", "activityName", "startTimeLocal", "distance", "duration"]
])

In [ ]:
# Localizamos la actividad por su identificador.
es_actividad = df_original["activityId"] == 14339983992

# Revisamos si Garmin la marca como registrada manualmente.
display(df_original.loc[
    es_actividad,
    ["activityName", "isManualActivity", "manualActivity"]
])

#### Revisión de los valores ausentes de ascenso

Hay 36 actividades sin elevationGain:
- 35 carreras en cinta.
- 1 carrera de exterior registrada manualmente: «Carrera 10Km»,
  del 10 de marzo de 2024.

Conservamos estas actividades y sus valores ausentes de ascenso.
No sustituimos los vacíos por cero, porque no disponer de una medición
no equivale a haber acumulado cero metros de subida.

La actividad manual conserva distancia y duración, por lo que puede
participar en los análisis de esas variables.

In [ ]:
# Comparamos la ubicación de los vacíos con la columna de ascenso.
for campo in campos_elevacion:
    mismos_vacios = (
        df_running[campo].isna()
        == df_running["elevationGain"].isna()
    ).all()

    print(campo, "¿Tiene los mismos vacíos que el ascenso?", mismos_vacios)

La comprobación fila por fila confirma que elevationLoss, minElevation
y maxElevation presentan ausencias en las mismas actividades que
elevationGain. Conservamos esos valores ausentes sin sustituirlos por cero.

avgElevation se revisará por separado, porque tiene una cobertura diferente.

In [ ]:
# Creamos una tabla auxiliar para investigar la disponibilidad.
# No modificamos df_running.
revision_potencia = pd.DataFrame({
    "anio": fechas_locales.dt.year,
    "tiene_potencia": df_running["avgPower"].notna()
})

# Contamos por año las actividades con y sin potencia media.
display(pd.crosstab(
    revision_potencia["anio"],
    revision_potencia["tiene_potencia"],
    colnames=["Potencia disponible"]
))

## Disponibilidad temporal de la potencia

La potencia media no está disponible en las actividades de 2023 y 2024.
En 2025 aparece en 18 de 104 actividades y en 2026, en las 87 registradas.

La disponibilidad cambia con el tiempo. La causa todavía no se ha confirmado.

Conservamos los valores ausentes y limitaremos los análisis de potencia
a las actividades que tengan esta información. No interpretaremos
su ausencia como potencia cero.

La propietaria confirma un cambio de reloj. Este cambio es una explicación
plausible de la aparición de datos de potencia; queda pendiente contrastar
la fecha de sustitución con el inicio de los registros.

In [ ]:
# Elegimos algunos campos con la misma cantidad de vacíos que avgPower.
campos_comparar = [
    "maxPower",
    "normPower",
    "minTemperature",
    "maxTemperature",
    "avgGroundContactTime",
    "aerobicTrainingEffect"
]

# Comparamos sus marcas de ausencia con las de potencia media.
for campo in campos_comparar:
    mismos_vacios = (
        df_running[campo].isna()
        == df_running["avgPower"].isna()
    ).all()

    print(campo, "¿Mismos vacíos que avgPower?", mismos_vacios)

#### Métricas con un patrón compartido de disponibilidad

maxPower, normPower, minTemperature, maxTemperature,
avgGroundContactTime y aerobicTrainingEffect presentan valores ausentes
en las mismas 212 actividades que avgPower.

Este patrón es compatible con el cambio de reloj confirmado por la
propietaria, aunque no se ha contrastado la fecha exacta del cambio.

Conservaremos las métricas y sus valores ausentes.
Sus análisis utilizarán únicamente las actividades con datos disponibles
e indicarán el periodo y el número de observaciones empleados.

In [ ]:
# Seleccionamos las actividades sin frecuencia cardiaca media.
sin_pulso = df_running["averageHR"].isna()

# Mostramos la actividad y otras métricas que podrían faltar también.
display(df_running.loc[
    sin_pulso,
    [
        "activityId",
        "activityName",
        "startTimeLocal",
        "averageHR",
        "maxHR",
        "averageRunningCadenceInStepsPerMinute",
        "steps"
    ]
])

#### Ausencias en la actividad manual

La actividad «Carrera 10Km», del 10 de marzo de 2024
(activityId: 14339983992), no contiene frecuencia cardiaca media
ni máxima, cadencia media ni pasos.

Cada uno de estos campos tiene un único valor ausente en el historial,
correspondiente a esta actividad registrada manualmente.

Conservamos la carrera para analizar distancia y duración.
No rellenaremos las métricas ausentes con ceros ni con promedios.
En los análisis que necesiten estas métricas, esta actividad
no aportará una observación válida.

In [ ]:
# Marcamos actividades con distanciony duracion menores o iguales a cero.
distancia_no_positiva = df_running["distance"] <= 0
duracion_no_positiva = df_running["duration"] <= 0

# Contamos cuántas actividades cumplen cada condición.
print("Distancia menor o igual a cero:", distancia_no_positiva.sum())
print("Duración menor o igual a cero:", duracion_no_positiva.sum())

# Mostramos las actividades que cumplen al menos una condición.
display(df_running.loc[
    distancia_no_positiva | duracion_no_positiva,
    ["activityId", "activityName", "startTimeLocal", "distance", "duration"]
])

In [ ]:
# Marcamos actividades cuya frecuencia cardiaca media supera la máxima.
pulso_incoherente = df_running["averageHR"] > df_running["maxHR"]

# Contamos los casos detectados.
print("Media de pulsaciones superior a la máxima:", pulso_incoherente.sum())

# Mostramos las actividades que incumplen esta relación.
display(df_running.loc[
    pulso_incoherente,
    ["activityId", "activityName", "averageHR", "maxHR"]
])

In [ ]:
# Detectamos si el cronometro supera el tiempo transcurrido.
cronometro_supera_total = (
    df_running["duration"] > df_running["elapsedDuration"]
)

# Detectamos si el tiempo en movimiento supera el transcurrido.
movimiento_supera_total = (
    df_running["movingDuration"] > df_running["elapsedDuration"]
)

# Mostramos cuántos casos hay de cada tipo.
print("Cronómetro mayor que transcurrido:", cronometro_supera_total.sum())
print("Movimiento mayor que transcurrido:", movimiento_supera_total.sum())

# Mostramos la que cumplen alguna condicion.
display(df_running.loc[
    cronometro_supera_total | movimiento_supera_total,
    [
        "activityId",
        "activityName",
        "duration",
        "elapsedDuration",
        "movingDuration"
    ]
])

#### Validaciones básicas de valores y coherencia

En el historial revisado:

- Las 317 actividades tienen distancia y duración presentes y positivas.
- No hay frecuencia cardiaca media superior a la máxima entre los registros
  que disponen de ambos valores.
- No hay duración del cronómetro superior al tiempo transcurrido.
- No hay tiempo en movimiento superior al transcurrido entre los registros
  con ambos valores disponibles.

Estas comprobaciones detectan inconsistencias concretas, pero no garantizan
por sí solas que las mediciones sean exactas.

Los valores ausentes se conservan y no se consideran registros validados.
No se ha eliminado ni corregido ninguna actividad en estas comprobaciones.

In [ ]:
# Elegimos la información necesaria para identificar cada actividad.
columnas_revision = [
    "activityId",
    "activityName",
    "startTimeLocal",
    "distance",
    "duration"
]

# Mostramos las cinco actividades con menor distancia.
print("DISTANCIAS MÁS CORTAS")
display(
    df_running.nsmallest(5, "distance")[columnas_revision]
)

# Mostramos las cinco actividades con mayor distancia.
print("DISTANCIAS MÁS LARGAS")
display(
    df_running.nlargest(5, "distance")[columnas_revision]
)

#### Revisión de distancias extremas

Se revisaron las cinco actividades de menor y mayor distancia.

El historial abarca aproximadamente de 1,68 km a 30 km.
La propietaria reconoce las actividades cortas como parte de sus inicios
y las tiradas largas como entrenamientos realizados.

No se detecta un motivo para corregir o eliminar estas actividades.
Un valor extremo no se considera un error únicamente por diferenciarse
del resto.

In [ ]:
# Mostramos las cinco actividades de menor duración.
print("DURACIONES MÁS CORTAS")
display(df_running.nsmallest(5, "duration")[columnas_revision])

# Mostramos las cinco actividades de mayor duración.
print("DURACIONES MÁS LARGAS")
display(df_running.nlargest(5, "duration")[columnas_revision])

In [ ]:
# Eliminamos espacios y saltos de línea al principio y al final del nombre.
# Conservamos los espacios interiores entre palabras.
df_running["activityName"] = df_running["activityName"].str.strip()

# Comprobamos el nombre de la actividad donde vimos el salto de línea.
nombre_revisado = df_running.loc[
    df_running["activityId"] == 14777933548,
    "activityName"
].iloc[0]

# repr permite ver caracteres ocultos, como \n, si todavía existen.
print(repr(nombre_revisado))

In [ ]:
# Extraemos la modalidad de running usando la función definida anteriorment.
df_running["modalidad"] = df_running["activityType"].apply(obtener_modalidad)

# Comprobamos el recuento de cada modalidad.
print(df_running["modalidad"].value_counts(dropna=False))

In [ ]:
# Añadimos a la tabla las fechas locales convertidas anteriormente.
df_running["fecha_local"] = fechas_locales

# Comprobamos su tipo y si hay valores ausentes.
print("Tipo:", df_running["fecha_local"].dtype)
print("Fechas ausentes:", df_running["fecha_local"].isna().sum())

In [ ]:
# Convertimos metros a kilometros y conservamos la columna original.
df_running["distance_km"] = df_running["distance"]/1000

# Mostramos ambas columnas para comprobar la conversión.
display(df_running[["distance", "distance_km"]].head())

In [ ]:
# Convertimos la duración de segundos a minutos.
# Conservamos duration con sus valores originales.
df_running["duration_min"] = df_running["duration"] / 60

# Comparamos la duración original y la convertida.
display(df_running[["duration", "duration_min"]].head())

In [ ]:
# Calculamos el ritmo medio utilizando el tiempo del cronómetro.
df_running["pace_min_km"] = (
    df_running["duration_min"] / df_running["distance_km"]
)

# Revisamos los datos utilizados y el ritmo resultante.
display(
    df_running[["distance_km", "duration_min", "pace_min_km"]].head()
)

#### Variables derivadas y unidades

Conservamos los campos originales y creamos:

| Variable | Cálculo | Unidad |
|---|---|---|
| distance_km | distance / 1000 | Kilómetros |
| duration_min | duration / 60 | Minutos |
| pace_min_km | duration_min / distance_km | Minutos por kilómetro |

El ritmo utiliza el tiempo del cronómetro (duration), no el tiempo
transcurrido ni el tiempo en movimiento.

Los minutos se almacenan en formato decimal: 5,5 min/km equivale
a 5:30 min/km. No redondeamos los valores utilizados en los cálculos.

In [ ]:
# Elegimos las columnas necesarias para interpretar cada ritmo.
columnas_ritmo = [
    "activityName",
    "fecha_local",
    "modalidad",
    "distance_km",
    "duration_min",
    "pace_min_km"
]

# Un ritmo menor significa correr más rápido.
print("CINCO RITMOS MÁS RÁPIDOS")
display(df_running.nsmallest(5, "pace_min_km")[columnas_ritmo])

# Un ritmo mayor significa tardar más por kilómetro.
print("CINCO RITMOS MÁS LENTOS")
display(df_running.nlargest(5, "pace_min_km")[columnas_ritmo])

In [ ]:
# Mostramos las columnas sus tipos y sus valores no vacios.
df_running.info(verbose=True,show_counts=True)

In [ ]:
# Creamos una tabla preparada sin el diccionario de modalidad.
# Su información seleccionada ya está en la columna modalidad.
df_preparado = df_running.drop(columns=["activityType"]).copy()

# Comprobamos que solo hemos quitado una columna, no actividades.
print("Tabla de trabajo:", df_running.shape)
print("Tabla preparada:", df_preparado.shape)

In [ ]:
# Definimos la carpeta para los datos preparados.
carpeta_processed = Path("../data/processed")

# Creamos la carpeta si no existe
carpeta_processed.mkdir(parents=True,exist_ok=True)

# Definimos un nombre que identifica la extracción de origen.
ruta_preparado = carpeta_processed / "running_preparado_20261031.csv"

# Guardamos la tabla sin añadir el indice de pandas como columna.
# mode="x" impide sobreescribir un archivo existente.
df_preparado.to_csv(
    ruta_preparado,
    index=False,
    encoding="utf-8",
    mode="x"
)

print("Tabla guardada en:", ruta_preparado.resolve())



#### Resultado de la preparación inicial

Se conserva un historial de 317 actividades:
- 282 carreras de exterior.
- 35 carreras en cinta.

La tabla preparada contiene 58 columnas. Se han limpiado los extremos
de los nombres, extraído la modalidad, convertido la fecha local y creado
variables de distancia en km, duración en minutos y ritmo en min/km.

No se han eliminado actividades ni rellenado valores ausentes.
Las métricas avanzadas conservan las limitaciones documentadas,
especialmente las relacionadas con el cambio de reloj.

#### Archivo generado

data/processed/running_preparado_20261031.csv

La lectura posterior confirma:
- 317 filas y 58 columnas.
- Los mismos identificadores y en el mismo orden.
- Los mismos nombres de columnas.

Esta verificación no compara todos los valores del archivo.
El JSON original se conserva sin modificar.